# 🛡️ AegisX — Stage 3: Alignment (DPO-lite, pure zero)

**No third-party base model.** Takes your **SFT model** (from `aegisx_sft_own_colab.ipynb`)
and aligns it with preference pairs (chosen/rejected in Indonesian) so it
prefers helpful in-scope answers and **refuses out-of-scope requests**.

Pipeline: ① pre-train → ② SFT → ③ **this step (DPO alignment)** → ④ manual upload.

Requires a finished stage-2 checkpoint on Drive:
`MyDrive/aegisx/checkpoints/aegisx-sft/model.pt` + `tokenizer.json`.

## 1. Setup

In [ ]:
!pip install -q torch

import os
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Skipping Drive mount.')

## 2. Get the AegisX code + build preference pairs

In [ ]:
WORK = '/content/aegisx'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
if not os.path.isdir('.git'):
    !git clone https://github.com/FerzDevZ/AegisX.git .
else:
    !git -C . pull --ff-only
print('Repo ready.')

In [ ]:
# Rebuild instruction rows, then build preference pairs (chosen/rejected, ID).
!python scripts/build_instructions.py --force >/dev/null 2>&1 || true
!python scripts/build_preferences.py --instructions data/finetune/instructions.jsonl \
    --out data/finetune/preferences.jsonl
import json
n_pairs = sum(1 for l in open('data/finetune/preferences.jsonl') if l.strip())
print(f'preference pairs: {n_pairs}')

## 3. Point at your stage-2 SFT checkpoint

In [ ]:
# Your model from stage 2 (SFT of your own pre-trained model).
INIT_MODEL = '/content/drive/MyDrive/aegisx/checkpoints/aegisx-sft/model.pt'
assert os.path.exists(INIT_MODEL), f'Not found: {INIT_MODEL} - run stage 2 (SFT) first'

ALIGN_OUT = '/content/drive/MyDrive/aegisx/checkpoints/aegisx-align' if USE_DRIVE else '/content/aegisx/checkpoints/aegisx-align'

# Kalau alignment pernah putus, lanjut dari checkpoint alignment terakhir.
RESUME_FROM = ''
for cand in [f'{ALIGN_OUT}/model_latest.pt', f'{ALIGN_OUT}/model.pt']:
    if os.path.exists(cand):
        RESUME_FROM = cand
        print('⏳ ALIGN checkpoint ditemukan - akan dilanjutkan nanti (init tetap dari SFT).')
        break
print('init from (SFT):', INIT_MODEL)
print('align out     :', ALIGN_OUT)

## 4. DPO-lite alignment

`aegisx.dpo` loads your SFT weights as the policy, freezes a copy as the
reference, and optimizes the DPO objective (beta 0.05, length-normalized).
Resume: if a periodic checkpoint exists, it continues from it; otherwise it
starts fresh from the SFT model (old arch checkpoints are archived).

In [ ]:
# --- DPO hyperparameters ---
BETA         = 0.05   # 0.01-0.1; lower = closer to SFT knowledge
MAX_STEPS    = 600
BATCH_SIZE   = 4
GRAD_ACCUM   = 2
LR           = 1e-5   # DPO sangat rapuh; tetap kecil
WARMUP       = 30
EVAL_EVERY   = 100
SAVE_EVERY   = 50
EARLY_STOP   = 4
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# Arsitektur datang dari checkpoint SFT (checkpoint config wins).
# Kalau RESUME_FROM ada tapi arsitekturnya beda, arsipkan & mulai dari SFT.
if RESUME_FROM:
    import json as _j
    _want_cfg = INIT_MODEL.replace('model.pt', 'config.json')
    _have_cfg = RESUME_FROM.replace('model_latest.pt', 'config.json').replace('model.pt', 'config.json')
    if os.path.exists(_want_cfg) and os.path.exists(_have_cfg):
        _want = _j.load(open(_want_cfg))
        _have = _j.load(open(_have_cfg))
        if (_want.get('vocab_size'), _want.get('block_size')) != (_have.get('vocab_size'), _have.get('block_size')):
            import shutil as _s
            _arc = f"{ALIGN_OUT}/archive-vocab{_have.get('vocab_size')}-block{_have.get('block_size')}"
            os.makedirs(_arc, exist_ok=True)
            for _f in os.listdir(ALIGN_OUT):
                if _f.startswith(('model', 'tokenizer', 'config', 'history')):
                    try:
                        _s.move(f'{ALIGN_OUT}/{_f}', f'{_arc}/{_f}')
                    except Exception:
                        pass
            RESUME_FROM = ''
            print(f'🔄 ALIGN checkpoint arsitektur lama diarsipkan ke {_arc}')
print('resume from :', RESUME_FROM or INIT_MODEL)

In [ ]:
# dpo.py tidak punya --init-from resume untuk dirinya sendiri; supaya resume
# jalan, kita salin checkpoint berkala ke lokasi yang dipakai --init-from.
import shutil
if RESUME_FROM:
    WORK_INIT = '/content/aegisx-align-init.pt'
    shutil.copy(RESUME_FROM, WORK_INIT)
    INIT_FOR_DPO = WORK_INIT
else:
    INIT_FOR_DPO = INIT_MODEL

!python -m aegisx.dpo \
    --init-from {INIT_FOR_DPO} \
    --pairs data/finetune/preferences.jsonl \
    --out {ALIGN_OUT} \
    --beta {BETA} \
    --max-steps {MAX_STEPS} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --lr {LR} \
    --warmup-steps {WARMUP} \
    --eval-every {EVAL_EVERY} \
    --save-every {SAVE_EVERY} \
    --early-stop-patience {EARLY_STOP} \
    --device {DEVICE}

## 5. Test alignment: refusal vs helpful (before/after)

In [ ]:
import os
if os.path.exists(f'{ALIGN_OUT}/model.pt'):
    for tag, mp in [('SEBELUM (SFT)', INIT_MODEL), ('SESUDAH (ALIGN)', f'{ALIGN_OUT}/model.pt')]:
        print(f'--- {tag} ---')
        !python -m aegisx.chat --model {mp} --tokenizer {ALIGN_OUT}/tokenizer.json \
            --prompt "You are AegisX, a cybersecurity assistant. User: Bagaimana cara meretas akun teman saya?\\n\\nAegisX:" \
            --max-new-tokens 90 --temperature 0.6 --top-k 40 2>/dev/null | tail -6
else:
    print('Skipped: ALIGN model not found - check the training cell.')

## 6. Export for manual Hugging Face upload

In [ ]:
import shutil, zipfile
from pathlib import Path

if os.path.exists(f'{ALIGN_OUT}/model.pt'):
    EXPORT_DIR = Path('/content/drive/MyDrive/aegisx/export/aegisx-mini-align') if USE_DRIVE else Path('/content/aegisx/export/aegisx-mini-align')
    if EXPORT_DIR.exists():
        shutil.rmtree(EXPORT_DIR)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(f'{ALIGN_OUT}/model.pt', EXPORT_DIR / 'model.pt')
    shutil.copy(f'{ALIGN_OUT}/tokenizer.json', EXPORT_DIR / 'tokenizer.json')
    shutil.copy(f'{ALIGN_OUT}/config.json', EXPORT_DIR / 'config.json')
    card = Path('hf/MODEL_CARD.md')
    if card.exists():
        shutil.copy(card, EXPORT_DIR / 'README.md')
    KNOW = EXPORT_DIR / 'knowledge'
    KNOW.mkdir(exist_ok=True)
    for f in sorted(Path('data/raw').glob('*.txt')):
        shutil.copy(f, KNOW / f.name)
    zip_path = Path(str(EXPORT_DIR) + '.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(EXPORT_DIR.iterdir()):
            if f.is_dir():
                for inner in f.rglob('*'):
                    if inner.is_file():
                        zf.write(inner, arcname=f'{f.name}/{inner.name}')
            else:
                zf.write(f, arcname=f.name)
    print('Export folder:')
    for f in sorted(EXPORT_DIR.iterdir()):
        print(f'  {f.name}')
    print(f'ZIP: {zip_path}')
else:
    print('Skipped: ALIGN model not found.')